In [4]:
# Vectorizer --> Bag of Words --> CountVectorizer
# Neural Network --> nn.Linear
# Activation Function --> GELU
# Regularization --> Dropout
# Optimizer --> Adam
# Loss Function --> BCEWithLogitsLoss

In [5]:
# =========================
# IMPORTS
# =========================
import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import joblib as jb


In [18]:
# =========================
# CONFIG (EASY TO CHANGE)
# =========================
EPOCHS = 10
BATCH_SIZE = 64

# Try multiple configs automatically
WIDTHS = [64,128,256,512,1024]
LAYERS = [1,2,3,4,6,8,10]

DROPOUT = 0.1
LEARNING_RATE = 1e-3
L2 = 1e-4  # weight decay

In [19]:
!nvidia-smi
%ls
print(torch.cuda.is_available())

Fri May 15 12:49:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.94                 Driver Version: 560.94         CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 Ti   WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   55C    P8             20W /  200W |    1153MiB /   8192MiB |      7%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [20]:
from google.colab import drive
drive.mount("/content/drive")

ModuleNotFoundError: No module named 'google'

In [21]:
# =========================
# LOAD DATA
# =========================
if torch.cuda.is_available():
    df = pd.read_csv("../../assets/cleaned_data/scikit_cleaned.csv")
else:
    df = pd.read_csv("/content/drive/MyDrive/scikit_cleaned.csv")

texts = (df['sender'].fillna('') + ' ' + df['receiver'].fillna('') + ' ' + df['date'].fillna('') + ' ' + df['subject'].fillna('') + ' ' + df['body'].fillna(''))
labels = df['label']

In [24]:
# =========================
# SPLIT: 60 / 20 / 20
# =========================
X_temp, X_test, y_temp, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42  # 0.25 of 80% = 20%
)

In [25]:
# =========================
# VECTORIZER (BAG OF WORDS)
# =========================
vectorizer = CountVectorizer(max_features=10000)

X_train_vec = vectorizer.fit_transform(X_train).toarray()
X_val_vec = vectorizer.transform(X_val).toarray()
X_test_vec = vectorizer.transform(X_test).toarray()

# Convert to tensors
X_train_tensor = torch.tensor(X_train_vec, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)

X_val_tensor = torch.tensor(X_val_vec, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test_vec, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

input_size = X_train_tensor.shape[1]

In [27]:
# =========================
# MODEL DEFINITION
# =========================
class PhishingNN(nn.Module):
    def __init__(self, input_size, width, depth):
        super().__init__()

        layers = []

        # input layer
        layers.append(nn.Linear(input_size, width))
        layers.append(nn.GELU())
        layers.append(nn.Dropout(DROPOUT))

        # hidden layers
        for _ in range(depth - 1):
            layers.append(nn.Linear(width, width))
            layers.append(nn.GELU())
            layers.append(nn.Dropout(DROPOUT))

        # output layer
        layers.append(nn.Linear(width, 1))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [28]:
# =========================
# TRAIN FUNCTION
# =========================
def train_model(model, X_train, y_train, X_val, y_val):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=L2)

    for epoch in range(EPOCHS):
        model.train()

        outputs = model(X_train).squeeze()
        loss = criterion(outputs, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Validation
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val).squeeze()
            val_preds = (torch.sigmoid(val_outputs) > 0.5).int()
            val_acc = accuracy_score(y_val, val_preds)

        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {loss.item():.4f} | Val Acc: {val_acc:.4f}")

    return val_acc


In [29]:
# =========================
# EXPERIMENT LOOP
# =========================
results = []

for width in WIDTHS:
    for depth in LAYERS:
        print(f"\nTraining model: width={width}, depth={depth}")

        model = PhishingNN(input_size, width, depth)

        val_acc = train_model(
            model,
            X_train_tensor,
            y_train_tensor,
            X_val_tensor,
            y_val_tensor
        )

        results.append({
            "width": width,
            "depth": depth,
            "val_acc": val_acc,
            "model": model
        })


Training model: width=64, depth=1
Epoch 1/10 | Loss: 0.6910 | Val Acc: 0.7853
Epoch 2/10 | Loss: 0.6150 | Val Acc: 0.8209
Epoch 3/10 | Loss: 0.5713 | Val Acc: 0.8577
Epoch 4/10 | Loss: 0.5317 | Val Acc: 0.8975
Epoch 5/10 | Loss: 0.4958 | Val Acc: 0.9312
Epoch 6/10 | Loss: 0.4634 | Val Acc: 0.9525
Epoch 7/10 | Loss: 0.4342 | Val Acc: 0.9604
Epoch 8/10 | Loss: 0.4072 | Val Acc: 0.9644
Epoch 9/10 | Loss: 0.3818 | Val Acc: 0.9665
Epoch 10/10 | Loss: 0.3581 | Val Acc: 0.9680

Training model: width=64, depth=2
Epoch 1/10 | Loss: 0.6946 | Val Acc: 0.5628
Epoch 2/10 | Loss: 0.6697 | Val Acc: 0.5773
Epoch 3/10 | Loss: 0.6478 | Val Acc: 0.6147
Epoch 4/10 | Loss: 0.6249 | Val Acc: 0.6765
Epoch 5/10 | Loss: 0.6021 | Val Acc: 0.7388
Epoch 6/10 | Loss: 0.5798 | Val Acc: 0.7872
Epoch 7/10 | Loss: 0.5560 | Val Acc: 0.8297
Epoch 8/10 | Loss: 0.5306 | Val Acc: 0.8687
Epoch 9/10 | Loss: 0.5028 | Val Acc: 0.9066
Epoch 10/10 | Loss: 0.4743 | Val Acc: 0.9356

Training model: width=64, depth=3
Epoch 1/10 | 

In [30]:
# =========================
# BEST MODEL SELECTION
# =========================
best = max(results, key=lambda x: x["val_acc"])

print("\nBest Model:")
print(best["width"], best["depth"], best["val_acc"])

best_model = best["model"]


Best Model:
1024 4 0.9818275550973063


In [16]:
def evaluate_on_test():
    best_model.eval()
    with torch.no_grad():
        outputs = best_model(X_test_tensor).squeeze()
        preds = (torch.sigmoid(outputs) > 0.5).int()
        acc = accuracy_score(y_test, preds)

    

    saved_metadata = {
        "width": best["width"],
        "depth": best["depth"],
        "dropout": DROPOUT,
        "input_size": input_size,
        "val_acc": best["val_acc"]
    }

    if torch.cuda.is_available():
        torch.save(best_model.state_dict(), "../../assets/models/deeplearning_model.pth")
        jb.dump(vectorizer, "../../assets/vectorizers/deeplearning_vectorizer.pkl")
        torch.save(saved_metadata, "../../assets/metadata/deeplearning_metadata.pth")
    else:
        torch.save(best_model.state_dict(), "/content/drive/MyDrive/PhishingModels/deeplearning_model.pth")
        jb.dump(vectorizer, "/content/drive/MyDrive/PhishingModels/deeplearning_vectorizer.pkl")
        torch.save(saved_metadata, "/content/drive/MyDrive/PhishingModels/deeplearning_metadata.pth")

    print(f"TEST ACCURACY: {acc:.4f}")
    print(confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds))

In [ ]:
#commented so I don't accidentally run

#evaluate_on_test()

TEST ACCURACY: 0.9672
[[7676  251]
 [ 258 7333]]
              precision    recall  f1-score   support

           0       0.97      0.97      0.97      7927
           1       0.97      0.97      0.97      7591

    accuracy                           0.97     15518
   macro avg       0.97      0.97      0.97     15518
weighted avg       0.97      0.97      0.97     15518

